In [49]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import os, glob
import ROOT
from scipy import stats

try:
#     plt.style.use('belle2')
    # plt.style.use('belle2_serif')
    plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style")   
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density, MC_stack_plot_weight

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
def do_random_choice_fast(df=None, random_state=251124):
    if df is None or df.empty:
        return pd.DataFrame()
    
    np.random.seed(random_state)
    
    # Add a random number column
    df_copied = df.copy()
    df_copied['_rand'] = np.random.rand(len(df_copied))
    
    # For each group, get the index of the row with max random number
    idx = df_copied.groupby(
        ['__experiment__', '__run__', '__event__', '__production__']
    )['_rand'].idxmax()
    
    final_df = df_copied.loc[idx].drop(columns='_rand')
    
    return final_df

In [51]:
def load_root_files(BCS_var, dir_topo_list, dir_list, tags, tree, cut, variables, bdt_cut):
    def find_files(tag):
        if tag == "ccbar":
            return [os.path.join(d, "standard.root") for d in dir_topo_list if os.path.exists(os.path.join(d, "standard.root"))]
        files = []
        for d in dir_list:
            pattern = os.path.join(d, tree, "no_bdt", str(bdt_cut), "weighted", f"weighted_{tag}*.root")
            files.extend(glob.glob(pattern))
        return sorted(files)

    common_extra_vars = [
        "Dp_chiProb", "BDT", "Pip_charge", "Pip_mcPDG", "Dp_isSignal", "etapip_Eta_isSignal",
        "Pip_genMotherPDG", "Pip_genMotherID", "etapip_Eta_genMotherPDG",
        "etapip_Eta_genMotherID", "__experiment__", "__run__", "__event__", "__production__", "BDT",
    ]
    ccbar_extra_vars = [
        "nAllSigCascDcyBr_7", "nAllSigCascDcyBr_3",
        "nAllSigCascDcyBr_9", "nAllSigCascDcyBr_13",
    ] + common_extra_vars

    def load_one_file(fp, extra_vars):
        df = get_pd(file=fp, tree=tree, base_filter=cut, variables=variables + extra_vars)
        if BCS_var == "BDT":
            return do_BCS(var="BDT", df=df, ascending=False)
        if BCS_var == "Dp_chiProb":
            return do_BCS(var="Dp_chiProb", df=df, ascending=False)
        if BCS_var == "random":
            return do_random_choice_fast(df=df)
        if BCS_var == "no_BCS":
            return df
        raise ValueError("BCS_var should be one of: 'BDT', 'Dp_chiProb', 'random', 'no_BCS'")

    file_map = {tag: find_files(tag) for tag in tags}
    for tag, files in file_map.items():
        print(f"{tag}: {len(files)} files found")

    dataframes = {}
    for tag in tags:
        files = file_map[tag]
        if not files:
            print(f"[Warning] No files found for tag = {tag}. Skipping.")
            continue

        extra_vars = ccbar_extra_vars if tag == "ccbar" else common_extra_vars
        dfs = [load_one_file(fp, extra_vars) for fp in files]
        df = pd.concat(dfs, ignore_index=True)

        if tag == "ccbar":
            mask_Dp_sig = df["Dp_isSignal"] == 1

            mask_Dsp_sig = (
                (df["etapip_Eta_isSignal"] == 1) &
                (df["Pip_genMotherID"] == df["etapip_Eta_genMotherID"]) &
                (
                    ((df["Pip_genMotherPDG"] == 431) & (df["Pip_mcPDG"] == 321)) |
                    ((df["Pip_genMotherPDG"] == -431) & (df["Pip_mcPDG"] == -321))
                )
            )

            mask_DpEtaPip_misID = (
                (
                    (df["Pip_charge"] == 1) &
                    (df["etapip_Eta_isSignal"] == 1) &
                    (df["Pip_genMotherID"] == df["etapip_Eta_genMotherID"]) &
                    (df["etapip_Eta_genMotherPDG"] == 411) &
                    (df["Pip_mcPDG"] == 211)
                ) |
                (
                    (df["Pip_charge"] == -1) &
                    (df["etapip_Eta_isSignal"] == 1) &
                    (df["Pip_genMotherID"] == df["etapip_Eta_genMotherID"]) &
                    (df["etapip_Eta_genMotherPDG"] == -411) &
                    (df["Pip_mcPDG"] == -211)
                )
            )

            mask_combinatorial = ~(mask_Dp_sig | mask_Dsp_sig | mask_DpEtaPip_misID)

            dataframes["ccbar_Dp_sig"] = df[mask_Dp_sig].copy().query(cut)
            dataframes["ccbar_Dsp_sig"] = df[mask_Dsp_sig].copy().query(cut)
            dataframes["ccbar_DpEtaPip_misID"] = df[mask_DpEtaPip_misID].copy().query(cut)
            dataframes["ccbar_combinatorial"] = df[mask_combinatorial].copy().query(cut)

        else:
            df.drop(columns="Dp_isSignal", inplace=True)
            dataframes[tag] = df.query(cut)

    return dataframes

In [52]:
# ================== MAIN SCRIPT ===================

plot_dir = "/media/jykim/T7/saved_plots/DRAW/etaKp/MC15rd_generic/gg"
os.makedirs(plot_dir, exist_ok=True)

# Settings
tree = "etapip_gg_K"
cut = "Dp_M>1.75 & Dp_M<2.045"
variables = ["Dp_M", "ds_weight", "rank_Dp_chiProb"]
var = ["Dp_M","rank_Dp_chiProb"]
xlabel = r"$M(\eta_{\gamma\gamma}K^+)$ [$\mathrm{GeV/c^2}$]"
bdt_cut = ""

bins = (1.75,2.045)
nbins = 59
# MC 설정
tags = ['ccbar', 'uubar', 'ddbar', 'ssbar', 'charged', 'mixed', 'taupair']

mc_base= "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/EtaHp/MC15rd_loose_v7_260108_nopi0veto"
mc_dirs = [
    f"{mc_base}/15rd_jae_e7_18_4S_v3",
    f"{mc_base}/15rd_jae_e20_b26_v1",
    f"{mc_base}/15rd_jae_e20_e26_4S_v2",
    f"{mc_base}/15rd_jae_e21_5S_scan_v1",
    f"{mc_base}/15rd_jae_mori_off_v1",
]
mc_base= "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/EtaHp/MC15rd_loose_v7_260108_nopi0veto/topo_no_bdt"
mc_topo_dirs = [
    f"{mc_base}/15rd_jae_e7_18_4S_v3/resultfile/result_{tree}",
    f"{mc_base}/15rd_jae_e20_b26_v1/resultfile/result_{tree}",
    f"{mc_base}/15rd_jae_e20_e26_4S_v2/resultfile/result_{tree}",
    f"{mc_base}/15rd_jae_e21_5S_scan_v1/resultfile/result_{tree}",
    f"{mc_base}/15rd_jae_mori_off_v1/resultfile/result_{tree}",
]

In [53]:
mc_data_no_BCS = load_root_files('no_BCS', mc_topo_dirs, mc_dirs, tags, tree, cut, variables, bdt_cut)


ccbar: 5 files found
uubar: 18 files found
ddbar: 11 files found
ssbar: 11 files found
charged: 12 files found
mixed: 12 files found
taupair: 11 files found


In [54]:
# mc_data_no_BCS

In [55]:
def summarize_bdt_reduction(dataframes, bdt_cut=0.86, bdt_var="BDT", weight_var="ds_weight"):
    rows = []

    ccbar_keys = [
        "ccbar_Dp_sig",
        "ccbar_Dsp_sig",
        "ccbar_DpEtaPip_misID",
        "ccbar_combinatorial",
    ]

    key_order = ccbar_keys + ["taupair", "mixed", "charged", "ssbar", "ddbar", "uubar"]

    def make_unweighted_row(category, df):
        n_before = len(df)
        df_after = df.query(f"{bdt_var} > {bdt_cut}")
        n_after = len(df_after)
        n_removed = n_before - n_after

        if n_before > 0:
            removed_frac = n_removed / n_before
            kept_frac = n_after / n_before
            removed_percent = 100 * removed_frac
            kept_percent = 100 * kept_frac
            removed_percent_err = 100 * np.sqrt(removed_frac * (1 - removed_frac) / n_before)
            kept_percent_err = removed_percent_err
        else:
            removed_percent = 0
            kept_percent = 0
            removed_percent_err = 0
            kept_percent_err = 0

        return {
            "category": category,
            "N before": n_before,
            "N after": n_after,
            "N removed": n_removed,
            "removed [%]": removed_percent,
            "removed err [%]": removed_percent_err,
            "kept [%]": kept_percent,
            "kept err [%]": kept_percent_err,
            "method": "unweighted",
        }

    def make_weighted_row(category, df):
        if weight_var not in df.columns:
            raise KeyError(f"{weight_var} is not found in dataframe for {category}")

        n_before = len(df)
        df_after = df.query(f"{bdt_var} > {bdt_cut}")
        n_after = len(df_after)
        n_removed = n_before - n_after

        w_before = df[weight_var]
        w_after = df_after[weight_var]

        sum_w_before = w_before.sum()
        sum_w_after = w_after.sum()
        sum_w_removed = sum_w_before - sum_w_after

        sum_w2_before = (w_before ** 2).sum()

        if sum_w_before > 0 and sum_w2_before > 0:
            removed_frac = sum_w_removed / sum_w_before
            kept_frac = sum_w_after / sum_w_before

            n_eff = sum_w_before ** 2 / sum_w2_before

            removed_percent = 100 * removed_frac
            kept_percent = 100 * kept_frac

            removed_percent_err = 100 * np.sqrt(removed_frac * (1 - removed_frac) / n_eff)
            kept_percent_err = removed_percent_err
        else:
            n_eff = 0
            removed_percent = 0
            kept_percent = 0
            removed_percent_err = 0
            kept_percent_err = 0

        return {
            "category": category,
            "N before": n_before,
            "N after": n_after,
            "N removed": n_removed,
            "weighted before": sum_w_before,
            "weighted after": sum_w_after,
            "weighted removed": sum_w_removed,
            "N_eff": n_eff,
            "removed [%]": removed_percent,
            "removed err [%]": removed_percent_err,
            "kept [%]": kept_percent,
            "kept err [%]": kept_percent_err,
            "method": f"weighted by {weight_var}",
        }

    for key, df in dataframes.items():
        if key == "ccbar_Dsp_sig":
            rows.append(make_weighted_row(key, df))
        else:
            rows.append(make_unweighted_row(key, df))

    summary = pd.DataFrame(rows)

    summary["category"] = pd.Categorical(summary["category"], categories=key_order, ordered=True)
    summary = summary.sort_values("category").reset_index(drop=True)

    print(f"\nBDT cut: {bdt_var} > {bdt_cut}\n")
    print("=== Category-wise reduction ===")
    print(summary.to_string(index=False, formatters={
        "weighted before": "{:.2f}".format,
        "weighted after": "{:.2f}".format,
        "weighted removed": "{:.2f}".format,
        "N_eff": "{:.2f}".format,
        "removed [%]": "{:.2f}".format,
        "removed err [%]": "{:.2f}".format,
        "kept [%]": "{:.2f}".format,
        "kept err [%]": "{:.2f}".format,
    }))

    return summary

In [56]:
summary_bdt = summarize_bdt_reduction(
    mc_data_no_BCS,
    bdt_cut=0.86,
    bdt_var="BDT",
    weight_var="ds_weight",
)


BDT cut: BDT > 0.86

=== Category-wise reduction ===
            category  N before  N after  N removed removed [%] removed err [%] kept [%] kept err [%]                method weighted before weighted after weighted removed    N_eff
        ccbar_Dp_sig      4816     2659       2157       44.79            0.72    55.21         0.72            unweighted             NaN            NaN              NaN      NaN
       ccbar_Dsp_sig     36340    14728      21612       56.06            0.27    43.94         0.27 weighted by ds_weight        36498.38       16038.66         20459.71 34046.15
ccbar_DpEtaPip_misID      2043     1120        923       45.18            1.10    54.82         1.10            unweighted             NaN            NaN              NaN      NaN
 ccbar_combinatorial    811722    36512     775210       95.50            0.02     4.50         0.02            unweighted             NaN            NaN              NaN      NaN
             taupair      5791       98       